# DINOv2 Multi-View v3 — 6-Slot Clinical MRI + SlotHead

### 相比 v2 的核心变更

| 维度 | v2 (昨天) | v3 (本次) |
|------|----------|----------|
| **输入** | 单视角 Sagittal T2 FS × 5 slices | 6 slot × 3 平面 (SAG/COR/AX, Fluid+T1) |
| **特征** | CLS token only (384-dim) | CLS + mean(patches) + focal_topk (1152-dim) |
| **Head** | SPA + CrossModalFusion + SliceTransformer + MLP | SlotHead: per-diagnosis attention over slots + anatomical priors |
| **图像** | 392×392 | 224×224 (DINOv2 原生, attention 快 9.4×) |
| **验证** | 56 gold studies (仅 Sagittal T2 FS) | ~200 gold studies (全部有任意 slot 的) |
| **速度** | ~1.5h/epoch | 预估 ~5-10min/epoch (224² + 简化架构) |

### 为什么多视角能解决过拟合

1. **信息量提升 6×**: ACL 需要 Sagittal, Baker's 需要 Axial, OA 需要 Coronal——单视角看不到的东西永远学不会
2. **SlotHead anatomical priors**: 模型内置先验——ACL 优先关注 Sagittal 序列, MCL 优先 Coronal, 不是从零学
3. **特征表达提升 3×**: CLS + mean + focal top-k 捕捉全局+平均+最强局部信号

### 与参考代码 (knee-rsna.ipynb) 的对齐

- ✅ 相同的 6 个 clinical slot 定义
- ✅ 相同的 SlotHead 架构 + anatomical priors
- ✅ 相同的特征提取 (CLS + mean + focal_topk)
- ✅ 相同的 224×224 输入 + 3-slice RGB窗口
- ✅ 相同的 DICOM 空间排序 + 侧位归一化
- ⚠️ 暂不包含物理 crop (PixelSpacing 依赖, 简化版)
- ⚠️ 暂不包含 TTA 重叠窗口 (仅训练, 非推理)



## 1. 环境安装


In [ ]:
!pip install -q timm pydicom opencv-python scikit-learn



## 2. 导入与配置


In [ ]:
from __future__ import annotations

import gc, math, os, sys, time, re
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
import pydicom
import cv2
from sklearn.metrics import roc_auc_score, f1_score

print(f'PyTorch {torch.__version__} | CUDA {torch.version.cuda}')
print(f'GPU count: {torch.cuda.device_count()}')



## 3. 配置


In [ ]:
# ============================================================
# Configuration — v3: Multi-View 6-Slot + SlotHead
# ============================================================

TARGET_COLUMNS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA',
    'Effusion', 'Synovitis', "Baker's",
    'Contusion', 'Fracture',
]

# Soft-label column names (reused from v2)
PROB_COLS   = [f'prob_{c}' for c in TARGET_COLUMNS]
WEIGHT_COLS = [f'weight_{c}' for c in TARGET_COLUMNS]
MASK_COLS   = [f'mask_{c}' for c in TARGET_COLUMNS]

# ---- 6 Clinical Slots (matching reference code) ----
# (name, plane, fluid, fatsat)
#   fluid=True → T2/PD (Fluid_Sensitive=1 in metadata)
#   fluid=False → T1-like structural (Fluid_Sensitive=0)
#   fatsat=True → Fat Suppression enabled
SLOTS = [
    ("SAG_FLUID_FS",   "Sagittal", True,  True),
    ("COR_FLUID_FS",   "Coronal",  True,  True),
    ("AX_FLUID_FS",    "Axial",    True,  True),
    ("SAG_FLUID_NOFS", "Sagittal", True,  False),
    ("COR_T1",         "Coronal",  False, False),
    ("SAG_T1",         "Sagittal", False, False),
]
N_SLOT = len(SLOTS)

# ---- Anatomical Priors for SlotHead ----
# Clinical knowledge: which planes each pathology is best seen on
# Slot indices: 0=SAG_FLUID_FS, 1=COR_FLUID_FS, 2=AX_FLUID_FS,
#               3=SAG_FLUID_NOFS, 4=COR_T1, 5=SAG_T1
SLOT_PRIORS = {
    "ACL":               (0, 3, 5),        # Sagittal views
    "MCL":               (1, 4),            # Coronal views
    "Medial Meniscus":   (0, 1, 3, 4),      # Sag + Cor
    "Lateral Meniscus":  (0, 1, 3, 4),
    "Medial OA":         (1, 4, 5),         # Cor + Sag T1
    "Lateral OA":        (1, 4, 5),
    "PF OA":             (0, 2, 5),         # Sag + Axial
    "Effusion":          (0, 2),            # Sag FS + Ax FS
    "Synovitis":         (0, 2),
    "Baker's":           (0,),              # Sag FS
    "Contusion":         (0, 1, 2),         # All FS planes
    "Fracture":          (0, 1, 2, 4, 5),   # All except Sag noFS
}

CFG = {
    # --- Paths (Kaggle) ---
    'comp_input':   '/kaggle/input/competitions/rsna-knee-abnormality-detection',
    'pseudo_input': '/kaggle/input/datasets/easoncyy/rsna-knee-soft-labels',
    'dicom_subdir': 'train_series',
    'output_dir':   '/kaggle/working',

    # --- Data ---
    'image_size': 224,          # DINOv2 native; 16×16 patches → 256 tokens (9.4× faster than 392)
    'cache_slices': 9,          # slices cached per slot
    'group_size': 3,            # 3 adjacent slices → RGB-like channels
    'center_pct': (0.2, 0.8),   # sample from central 60% of slice stack

    # --- Training subset control ---
    # Set to None to use ALL studies. Set to N to randomly sample N studies for faster iteration.
    'train_studies_limit': 500,   # smaller = faster; increase for final run

    # --- Model ---
    'dinov2_variant': 'vit_small_patch14_dinov2.lvd142m',
    'cls_dim': 384,             # DINOv2 ViT-S hidden dim
    'feature_dim': 384 * 3,     # CLS + mean(patches) + focal_topk
    'slot_hidden': 256,         # SlotHead hidden dim
    'num_classes': 12,

    # --- Unfreeze strategy (matching reference) ---
    'unfreeze_layers': 6,       # last 6 of 12 DINOv2 blocks + final norm

    # --- Training ---
    'batch_size': 6,            # 6 studies × 6 slots = 36 DINOv2 forwards/step
    'grad_accum_steps': 2,      # effective batch = 6 × 2GPUs × 2 = 24
    'epochs': 50,
    'lr': 2e-4,
    'backbone_lr': 1e-5,
    'weight_decay': 1e-4,
    'lr_t0': 15,
    'lr_t_mult': 2,
    'lr_eta_min': 1e-6,
    'dropout': 0.2,
    'grad_clip': 1.0,
    'early_stop_patience': 10,
    'mixed_precision': True,
    'num_workers': 2,
}

# Device setup
N_GPUS = torch.cuda.device_count()
DEVICE = torch.device('cuda' if N_GPUS > 0 else 'cpu')
IS_MAIN = True

if IS_MAIN:
    print(f'GPUs: {N_GPUS} | Device: {DEVICE}')
    print(f'--- v3: Multi-View 6-Slot + SlotHead ---')
    for k, v in CFG.items():
        print(f'  {k}: {v}')



## 4. Slot 匹配


In [ ]:
# ============================================================
# v3: Slot Matching — map DICOM series to 6 clinical slots
# ============================================================
#
# Uses train_series.csv metadata (Anatomical_Plane, Fluid_Sensitive,
# Fat_Suppression) to assign the best series for each clinical slot.
# Falls back to SeriesDescription heuristics when metadata is missing.

def match_slots_for_study(study_series_df):
    """Assign one series per slot for a single study.

    Args:
        study_series_df: DataFrame subset for ONE study, with columns:
            SeriesInstanceUID, Anatomical_Plane, Fluid_Sensitive,
            Fat_Suppression, n_slices (pre-computed), dir (DICOM path)

    Returns:
        dict: {slot_name: dict(series_uid, dir, n_slices, plane) or None}
    """
    slots_found = {}
    for slot_name, plane, fluid, fatsat in SLOTS:
        candidates = study_series_df[
            (study_series_df['Anatomical_Plane'] == plane) &
            (study_series_df['Fluid_Sensitive'] == (1 if fluid else 0)) &
            (study_series_df['Fat_Suppression'] == (1 if fatsat else 0))
        ]

        # Fallback for structural slots (T1): relax fatsat requirement
        if len(candidates) == 0 and not fluid:
            candidates = study_series_df[
                (study_series_df['Anatomical_Plane'] == plane) &
                (study_series_df['Fluid_Sensitive'] == 0)
            ]

        if len(candidates) > 0:
            # Pick series with most slices
            best = candidates.sort_values('n_slices', ascending=False).iloc[0]
            slots_found[slot_name] = {
                'series_uid': best['SeriesInstanceUID'],
                'dir': best['dir'],
                'n_slices': int(best['n_slices']),
                'plane': plane,
            }
        else:
            slots_found[slot_name] = None

    return slots_found


def build_study_slot_map(series_meta, dicom_root):
    """Build slot→series mapping for all studies.

    Args:
        series_meta: DataFrame from train_series.csv
        dicom_root: Path to DICOM directory root

    Returns:
        tuple:
            slot_map: {study_uid: {slot_name: dict(series_uid, dir, n_slices, plane) or None}}
            study_series_map: {study_uid: DataFrame with columns incl. dir, n_slices, SeriesInstanceUID}
    """
    df = series_meta.copy()
    df['StudyInstanceUID'] = df['StudyInstanceUID'].astype(str)
    df['SeriesInstanceUID'] = df['SeriesInstanceUID'].astype(str)

    # Pre-compute DICOM directory and slice count
    dirs = []
    n_slices_list = []
    for _, row in df.iterrows():
        d = str(dicom_root / row['StudyInstanceUID'] / row['SeriesInstanceUID'])
        dirs.append(d)
        if os.path.isdir(d):
            n_slices_list.append(len([f for f in os.listdir(d) if f.endswith('.dcm')]))
        else:
            n_slices_list.append(0)
    df['dir'] = dirs
    df['n_slices'] = n_slices_list

    # Ensure required columns
    for col in ['Fluid_Sensitive', 'Fat_Suppression', 'Anatomical_Plane']:
        if col not in df.columns:
            raise KeyError(f'train_series.csv missing column: {col}')

    slot_map = {}
    study_series_map = {}

    for study_uid, grp in df.groupby('StudyInstanceUID'):
        study_series_map[study_uid] = grp
        slot_map[study_uid] = match_slots_for_study(grp)

    # Statistics
    slot_counts = {}
    for slots in slot_map.values():
        for name, sid in slots.items():
            slot_counts[name] = slot_counts.get(name, 0) + (1 if sid is not None else 0)

    if IS_MAIN:
        n_studies = len(slot_map)
        print(f'Slot map: {n_studies} studies')
        for name, count in slot_counts.items():
            print(f'  {name:<18s}: {count:5d}/{n_studies} ({count/n_studies*100:.0f}%)')

    return slot_map, study_series_map



## 5. DICOM 读取


In [ ]:
# ============================================================
# v3: DICOM I/O — spatial sorting, normalization, 9-slice sampling
# ============================================================

PLANE_SORT_AXIS = {'Sagittal': 0, 'Coronal': 1, 'Axial': 2}

_DICOM_SPECIFIC_TAGS = [
    (0x0020, 0x0032),  # ImagePositionPatient
    (0x0020, 0x0013),  # InstanceNumber
    (0x0028, 0x0002),  # SamplesPerPixel
    (0x0028, 0x0004),  # PhotometricInterpretation
    (0x0028, 0x0010),  # Rows
    (0x0028, 0x0011),  # Columns
    (0x0028, 0x0100),  # BitsAllocated
    (0x0028, 0x0101),  # BitsStored
    (0x0028, 0x0102),  # HighBit
    (0x0028, 0x0103),  # PixelRepresentation
    (0x0028, 0x0030),  # PixelSpacing
    (0x0028, 0x1052),  # RescaleIntercept
    (0x0028, 0x1053),  # RescaleSlope
    (0x7FE0, 0x0010),  # PixelData
]

def _get_slice_position(ds, plane=None):
    try:
        ipp = getattr(ds, 'ImagePositionPatient', None)
        if ipp and len(ipp) >= 3:
            axis = PLANE_SORT_AXIS.get(plane, 2) if plane else 2
            return float(ipp[axis])
    except: pass
    try:
        sl = getattr(ds, 'SliceLocation', None)
        if sl is not None: return float(sl)
    except: pass
    try: return float(getattr(ds, 'InstanceNumber', 0))
    except: return 0.0


def read_series_volume(series_dir, plane=None, image_size=224):
    """Read and normalize all slices in a DICOM series.

    Returns:
        volume: np.ndarray [N_slices, H, W] float32 in [0, 1]
        px: PixelSpacing or None
    """
    series_dir = Path(series_dir)
    dcm_paths = sorted(series_dir.glob('*.dcm'))
    if not dcm_paths:
        dcm_paths = sorted(series_dir.glob('*'))
    if not dcm_paths:
        raise RuntimeError(f'No DICOM files: {series_dir}')

    slices_info = []
    px = None
    for p in dcm_paths:
        try:
            ds = pydicom.dcmread(str(p), force=True, specific_tags=_DICOM_SPECIFIC_TAGS)
            pos = _get_slice_position(ds, plane)
            img = ds.pixel_array.astype(np.float32)
            # Rescale
            slope = float(getattr(ds, 'RescaleSlope', 1) or 1)
            intercept = float(getattr(ds, 'RescaleIntercept', 0) or 0)
            img = img * slope + intercept
            slices_info.append((pos, img))
            if px is None:
                try:
                    ps = getattr(ds, 'PixelSpacing', None)
                    if ps and len(ps) >= 2:
                        px = float(ps[0])
                except: pass
        except Exception:
            continue

    if not slices_info:
        raise RuntimeError(f'No readable DICOM: {series_dir}')

    slices_info.sort(key=lambda x: x[0])
    images = np.stack([img for _, img in slices_info], axis=0)

    # Robust percentile normalization
    v_low = np.percentile(images, 1.0)
    v_high = np.percentile(images, 99.0)
    images = np.clip(images, v_low, v_high)
    denom = max(v_high - v_low, 1e-6)
    images = (images - v_low) / denom

    # Resize
    resized = []
    for img in images:
        r = cv2.resize(img, (image_size, image_size), interpolation=cv2.INTER_LINEAR)
        resized.append(r)
    return np.stack(resized, axis=0).astype(np.float32), px


def sample_cache_slices(volume, n_cache=9, center_pct=(0.2, 0.8)):
    """Sample N slices uniformly from the central portion of the stack.

    Matching reference: linspace(low_idx, high_idx, n_cache).
    """
    n_total = volume.shape[0]
    if n_total <= n_cache:
        # Repeat last slice if needed
        indices = list(range(n_total))
        while len(indices) < n_cache:
            indices.append(indices[-1])
        return volume[np.array(indices)]

    low = int(center_pct[0] * (n_total - 1))
    high = int(center_pct[1] * (n_total - 1))
    if high <= low:
        low, high = 0, n_total - 1
    indices = np.unique(np.linspace(low, high, n_cache).astype(int))
    while len(indices) < n_cache:
        indices = np.append(indices, indices[-1])
    return volume[indices[:n_cache]]


def normalise_laterality(image, plane, laterality):
    """Mirror right knees to left-knee convention (reference code pattern)."""
    if laterality != 'R':
        return image
    # image: [H, W] or [N, H, W]
    if plane in ('Coronal', 'Axial'):
        return np.flip(image, axis=-1)
    return np.flip(image, axis=-2)  # Sagittal: flip H



## 6. 模型 — SlotHead + MultiViewModel


In [ ]:
# ============================================================
# v3: SlotHead + MultiViewModel — matching reference architecture
# ============================================================

class SlotHead(nn.Module):
    """Per-diagnosis attention over MRI slots with anatomical priors.

    Matching reference code (knee-rsna.ipynb):
      - Projects slot features: LayerNorm → Linear → GELU
      - Learned slot embedding + per-diagnosis query vectors
      - Anatomical prior biases attention (soft, additive, 0.55)
      - Mask fills missing slots with -1e4 before softmax
    """

    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2):
        super().__init__()
        self.proj = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, hidden),
            nn.GELU(),
        )
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden

        # Anatomical prior: which slots each diagnosis prefers
        prior = torch.zeros(n_out, n_slot)
        for target_name, slot_indices in SLOT_PRIORS.items():
            if target_name in TARGET_COLUMNS:
                prior[TARGET_COLUMNS.index(target_name), list(slot_indices)] = 0.55
        self.register_buffer("slot_prior", prior)

    def forward(self, x, mask):
        """x: [B, S, D]  slot features
           mask: [B, S]  1=present, 0=missing
        Returns: [B, n_out] logits
        """
        h = self.proj(x) + self.slot_emb                              # [B, S, H]
        attention = (
            torch.einsum("bsh,oh->bos", h, self.query)                # [B, n_out, S]
            / math.sqrt(self.hidden)
            + self.slot_prior.unsqueeze(0)                             # add anatomical bias
        )
        attention = attention.masked_fill(
            mask.unsqueeze(1) < 0.5, -1e4
        ).softmax(-1)                                                  # [B, n_out, S]
        context = self.drop(torch.einsum("bos,bsh->boh", attention, h))  # [B, n_out, H]
        return (context * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias


class MultiViewModel(nn.Module):
    """DINOv2 + SlotHead for multi-view knee MRI.

    Data flow:
      Input:  [B, 6, 3, 224, 224]  (batch, slots, RGB-channels, H, W)
      → DINOv2 per-slot: [B*6, 3, 224, 224] → [B*6, 257, 384]
      → CLS[0] + mean(patches) + focal_topk(patches) → [B*6, 1152]
      → Reshape: [B, 6, 1152]
      → SlotHead: [B, 6, 1152] × mask → [B, 12]

    Matching reference code:
      - CLS token + mean patch + focal top-k (12.5% of patches)
      - ImageNet normalization
      - Partial DINOv2 unfreeze
    """

    def __init__(self, dinov2_model, n_slots=6, cls_dim=384,
                 n_classes=12, slot_hidden=256, dropout=0.2,
                 unfreeze_layers=6):
        super().__init__()
        self.n_slots = n_slots
        self.cls_dim = cls_dim
        self.feature_dim = cls_dim * 3  # CLS + mean + focal
        self.unfreeze_layers = unfreeze_layers

        self.dinov2 = dinov2_model

        # Partial unfreeze (matching reference)
        n_blocks = len(self.dinov2.blocks)
        if unfreeze_layers > 0:
            for p in self.dinov2.parameters():
                p.requires_grad = False
            unfreeze_start = max(0, n_blocks - unfreeze_layers)
            for block in self.dinov2.blocks[unfreeze_start:]:
                for p in block.parameters():
                    p.requires_grad = True
            if hasattr(self.dinov2, 'norm'):
                for p in self.dinov2.norm.parameters():
                    p.requires_grad = True
            trainable_dino = sum(p.numel() for p in self.dinov2.parameters() if p.requires_grad)
            total_dino = sum(p.numel() for p in self.dinov2.parameters())
            if IS_MAIN:
                print(f'[DINOv2] Blocks {unfreeze_start}-{n_blocks-1} UNFROZEN '
                      f'({trainable_dino/1e6:.1f}M / {total_dino/1e6:.1f}M params)')

        # SlotHead
        self.head = SlotHead(
            dim=self.feature_dim, n_slot=n_slots, n_out=n_classes,
            hidden=slot_hidden, p=dropout,
        )

        # ImageNet normalization (matching reference)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

        # Summary
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        if IS_MAIN:
            print(f'[MultiViewModel] Total: {total/1e6:.1f}M | '
                  f'Trainable: {trainable/1e6:.1f}M ({trainable/total*100:.0f}%)')

    def _extract_features(self, x_3ch):
        """Extract CLS + mean + focal_topk from DINOv2 features.

        x_3ch: [B_total, 3, H, W] float32, already normalized
        Returns: [B_total, feature_dim]
        """
        if self.unfreeze_layers > 0:
            features = self.dinov2.forward_features(x_3ch)  # [B, N+1, D]
        else:
            with torch.no_grad():
                features = self.dinov2.forward_features(x_3ch)

        cls = features[:, 0, :]                              # [B, D]
        patches = features[:, 1:, :]                         # [B, N, D]
        mean_p = patches.mean(dim=1)                         # [B, D]
        k = max(1, patches.shape[1] // 8)                    # top 12.5%
        focal = patches.topk(k, dim=1).values.mean(dim=1)    # [B, D]
        return torch.cat([cls, mean_p, focal], dim=1)        # [B, 3*D]

    def forward(self, images, mask):
        """images: [B, S, 3, H, W] uint8 in [0, 255]
           mask:   [B, S] float32, 1=present, 0=missing
        Returns: [B, n_classes] logits
        """
        B, S = images.shape[:2]

        # Flatten slots → batch dimension
        x = images.reshape(B * S, 3, images.shape[-2], images.shape[-1])
        x = x.float().div_(255.0)
        x = (x - self.mean) / self.std

        # DINOv2 feature extraction
        features = self._extract_features(x)                 # [B*S, feature_dim]
        features = features.reshape(B, S, -1)                # [B, S, feature_dim]

        # SlotHead with mask
        return self.head(features, mask)                     # [B, n_classes]

    def train(self, mode=True):
        super().train(mode)
        # DINOv2 stays in eval mode (deterministic DropPath/norm)
        # but parameters still receive gradients when unfrozen
        self.dinov2.eval()
        return self



## 7. 损失函数


In [ ]:
# ============================================================
# v3: Weighted Soft BCE Loss (reused from v2, unchanged)
# ============================================================

class WeightedSoftBCELoss(nn.Module):
    """BCE loss with soft probability targets and per-class reliability weights.

    Shape:
        logits:       [B, C]  model output logits
        prob_targets: [B, C]  calibrated probabilities (0.01 ~ 0.99)
        weights:      [B, C]  per-class training weights (0.05 ~ 1.0)
        masks:        [B, C]  binary mask (0=ignore this class for this sample)
    """

    def __init__(self, eps: float = 1e-7):
        super().__init__()
        self.eps = eps

    def forward(self, logits, prob_targets, weights, masks):
        targets = prob_targets.clamp(self.eps, 1.0 - self.eps)
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        weighted = bce * weights * masks
        denom = masks.sum().clamp(min=1)
        return weighted.sum() / denom


class HardBCELoss(nn.Module):
    """Standard BCE for gold-labeled studies."""
    def forward(self, logits, targets):
        return F.binary_cross_entropy_with_logits(logits, targets)



## 8. 数据集


In [ ]:
# ============================================================
# v3: MultiViewDataset — 6-slot clinical MRI
# ============================================================

class MultiViewDataset(Dataset):
    """Multi-view knee MRI dataset with 6 clinical slots.

    Each study returns:
      - slots: [6, 3, 224, 224] uint8 tensor (3 adjacent slices as RGB-like channels)
      - mask: [6] float32 (1=slot present, 0=missing)
      - labels or soft_label info depending on is_train and label_type

    Training data augmentation:
      - Random window selection from cached 9 slices
    Validation:
      - Fixed middle window
    """

    def __init__(self, study_uids, slot_map, cache, mask_array,
                 labels_df, study_index, is_train=True):
        self.study_uids = list(study_uids)
        self.slot_map = slot_map
        self.cache = cache
        self.mask_array = mask_array
        self.labels_df = labels_df
        self.study_index = study_index
        self.is_train = is_train

        # Filter: keep only studies that are in the cache
        valid_uids = []
        skipped = 0
        for uid in self.study_uids:
            if uid in self.study_index:
                valid_uids.append(uid)
            else:
                skipped += 1
        self.study_uids = valid_uids
        if IS_MAIN and skipped:
            print(f'[{type(self).__name__}] {skipped} studies skipped (not in cache)')

    def __len__(self):
        return len(self.study_uids)

    def __getitem__(self, idx):
        uid = self.study_uids[idx]
        row_idx = self.study_index[uid]

        # Load from cache
        slots = torch.from_numpy(self.cache[row_idx].copy())  # [6, 9, 224, 224]
        mask = torch.from_numpy(self.mask_array[row_idx].copy())  # [6]

        # Select 3-slice window
        n_slices = slots.shape[1]  # 9
        if self.is_train:
            max_start = n_slices - 3
            start = torch.randint(0, max_start + 1, (1,)).item() if max_start > 0 else 0
        else:
            start = (n_slices - 3) // 2  # middle window

        window = slots[:, start:start+3]  # [6, 3, 224, 224]

        # Labels
        label_row = self.labels_df.loc[uid]

        if self.is_train:
            # Unified soft-label format: gold studies have prob=hard_label, weight=1, mask=1
            probs = torch.tensor(
                [float(label_row.get(c, 0.5)) for c in PROB_COLS], dtype=torch.float32)
            weights = torch.tensor(
                [float(label_row.get(c, 0.1)) for c in WEIGHT_COLS], dtype=torch.float32)
            soft_masks = torch.tensor(
                [float(label_row.get(c, 0.0)) for c in MASK_COLS], dtype=torch.float32)
            return {
                'slots': window,
                'mask': mask,
                'prob_targets': probs,
                'weights': weights,
                'soft_masks': soft_masks,
                'study_uid': uid,
            }
        else:
            # Validation: hard labels + masks for partial-label support
            labels = torch.tensor(
                [float(label_row.get(c, 0.0)) for c in TARGET_COLUMNS], dtype=torch.float32)
            val_masks = torch.tensor(
                [float(label_row.get(c, 0.0)) for c in MASK_COLS], dtype=torch.float32)
            return {
                'slots': window,
                'mask': mask,
                'labels': labels,
                'val_masks': val_masks,
                'study_uid': uid,
            }



## 9. 加载数据与标签


In [ ]:
# ============================================================
# v3: Load metadata + pseudo labels
# ============================================================

comp_input = Path(CFG['comp_input'])
pseudo_input = Path(CFG['pseudo_input'])

# Load competition metadata
train_meta = pd.read_csv(comp_input / 'train.csv')
train_meta['StudyInstanceUID'] = train_meta['StudyInstanceUID'].astype(str)
series_meta = pd.read_csv(comp_input / 'train_series.csv')
series_meta['StudyInstanceUID'] = series_meta['StudyInstanceUID'].astype(str)
series_meta['SeriesInstanceUID'] = series_meta['SeriesInstanceUID'].astype(str)

# Split gold vs unlabeled
# Use .any(axis=1) instead of .all(axis=1): a study is "gold" if it has
# at least one human-annotated label. Unlabeled targets are masked out
# (mask=0) in training loss and validation AUC.
label_cols_present = [c for c in TARGET_COLUMNS if c in train_meta.columns]
has_any_label = train_meta[label_cols_present].notna().any(axis=1)
gold_df = train_meta[has_any_label].copy()
unlabeled_df = train_meta[~has_any_label].copy()

# Split gold into train/val (80/20)
gold_studies = gold_df['StudyInstanceUID'].values
np.random.seed(42)
np.random.shuffle(gold_studies)
n_val = int(len(gold_studies) * 0.2)
val_gold_uids = set(gold_studies[:n_val])
train_gold_uids = set(gold_studies[n_val:])

if IS_MAIN:
    n_labeled_per_study = gold_df[label_cols_present].notna().sum(axis=1)
    print(f'Gold studies (any label): {len(gold_df)} '
          f'(avg {n_labeled_per_study.mean():.1f} labels/study)')
    print(f'  Train gold: {len(train_gold_uids)} | Val gold: {len(val_gold_uids)}')
    print(f'Unlabeled studies: {len(unlabeled_df)}')

# Load calibrated pseudo-labels
calibrated_df = pd.read_csv(pseudo_input / 'pseudo_labels_calibrated.csv')
calibrated_df['StudyInstanceUID'] = calibrated_df['StudyInstanceUID'].astype(str)

# Build soft-label DataFrame (for pseudo-labeled studies, used in training)
pseudo_labels = calibrated_df[['StudyInstanceUID']].copy()
for c in PROB_COLS:
    pseudo_labels[c] = calibrated_df[c]
for c in WEIGHT_COLS:
    pseudo_labels[c] = calibrated_df[c]
for c in MASK_COLS:
    pseudo_labels[c] = calibrated_df[c]
pseudo_labels = pseudo_labels.set_index('StudyInstanceUID')
for c in PROB_COLS + WEIGHT_COLS + MASK_COLS:
    pseudo_labels[c] = pd.to_numeric(pseudo_labels[c], errors='coerce').fillna(
        0.5 if 'prob' in c else 0.1).astype(np.float32)

# Build hard-label DataFrame (for gold studies, used in training + validation)
# Keep NaN as NaN — we use them to build per-target masks.
# "Labeled negative" (0.0) and "unlabeled" (NaN) are different things.
gold_labels = gold_df[['StudyInstanceUID'] + label_cols_present].copy()
gold_labels = gold_labels.set_index('StudyInstanceUID')
for c in TARGET_COLUMNS:
    if c not in gold_labels.columns:
        gold_labels[c] = np.nan
gold_labels = gold_labels.apply(pd.to_numeric, errors='coerce')

# Training set:
#   - Gold studies in train split → hard labels
#   - All unlabeled studies → soft labels
# Combine into one labels DataFrame with both hard and soft columns
# We'll distinguish in the dataset via a flag

# For training: use train_gold_uids (hard) + unlabeled (soft)
train_study_uids = list(train_gold_uids) + list(unlabeled_df['StudyInstanceUID'].unique())

# For validation: use val_gold_uids (hard labels only)
val_study_uids = list(val_gold_uids)

# Training labels: gold rows get hard labels + dummy soft cols; pseudo rows get soft labels
# Build unified labels_df
all_train_rows = []
for uid in train_gold_uids:
    if uid not in gold_labels.index:
        continue
    row = {'StudyInstanceUID': uid}
    # Gold labels → "soft" format with per-target mask
    #   labeled target:   prob=hard_label, weight=1.0, mask=1.0
    #   unlabeled target: prob=0.5, weight=0.0, mask=0.0 (skipped in loss)
    for c in TARGET_COLUMNS:
        raw = gold_labels.loc[uid, c]
        is_labeled = not pd.isna(raw)
        hard_val = float(raw) if is_labeled else 0.0
        row[c] = hard_val
        row[f'prob_{c}'] = max(0.01, min(0.99, hard_val)) if is_labeled else 0.5
        row[f'weight_{c}'] = 1.0 if is_labeled else 0.0
        row[f'mask_{c}'] = 1.0 if is_labeled else 0.0
    row['is_gold'] = True
    all_train_rows.append(row)

for uid in unlabeled_df['StudyInstanceUID'].unique():
    if uid not in pseudo_labels.index:
        continue
    row = {'StudyInstanceUID': uid}
    for c in TARGET_COLUMNS:
        row[c] = 0.0  # dummy, not used
    for c in PROB_COLS:
        row[c] = float(pseudo_labels.loc[uid, c])
    for c in WEIGHT_COLS:
        row[c] = float(pseudo_labels.loc[uid, c])
    for c in MASK_COLS:
        row[c] = float(pseudo_labels.loc[uid, c])
    row['is_gold'] = False
    all_train_rows.append(row)

train_labels = pd.DataFrame(all_train_rows).set_index('StudyInstanceUID')
# Build val_labels with mask columns (same format as train, so dataset can
# return val_masks for partial-label validation).
val_labels_raw = gold_labels[gold_labels.index.isin(val_gold_uids)].copy()
val_rows = []
for uid in val_labels_raw.index:
    row = {'StudyInstanceUID': uid}
    for c in TARGET_COLUMNS:
        raw = val_labels_raw.loc[uid, c]
        is_labeled = not pd.isna(raw)
        row[c] = float(raw) if is_labeled else 0.0
        row[f'prob_{c}'] = 0.0   # unused in val, placeholder
        row[f'weight_{c}'] = 0.0  # unused in val, placeholder
        row[f'mask_{c}'] = 1.0 if is_labeled else 0.0
    row['is_gold'] = True
    val_rows.append(row)
val_labels = pd.DataFrame(val_rows).set_index('StudyInstanceUID')

# Training subset limit (for faster iteration)
if CFG['train_studies_limit'] and CFG['train_studies_limit'] < len(train_labels):
    subset_uids = np.random.choice(
        train_labels.index.values, CFG['train_studies_limit'], replace=False)
    train_labels = train_labels.loc[subset_uids]
    if IS_MAIN:
        print(f'Training subset: {len(train_labels)} studies (limit={CFG["train_studies_limit"]})')

if IS_MAIN:
    n_gold_train = train_labels['is_gold'].sum() if 'is_gold' in train_labels.columns else 0
    train_gold_masks = train_labels[train_labels['is_gold']][MASK_COLS].values if n_gold_train > 0 else np.zeros((0, 12))
    train_labeled_pct = train_gold_masks.mean() * 100 if n_gold_train > 0 else 0
    print(f'\nTrain studies: {len(train_labels):,}  '
          f'(gold={n_gold_train}, pseudo={len(train_labels) - n_gold_train})')
    if n_gold_train > 0:
        print(f'  Gold train label coverage: {train_labeled_pct:.0f}% of targets labeled (partial labels)')

    val_masks = val_labels[MASK_COLS].values
    val_labeled_per_class = val_masks.sum(axis=0)
    print(f'Val studies:   {len(val_labels):,}')
    print(f'  Labeled per class: min={int(val_labeled_per_class.min())}, '
          f'max={int(val_labeled_per_class.max())}, '
          f'mean={val_labeled_per_class.mean():.1f}')



## 10. 构建 RAM 缓存


In [ ]:
# ============================================================
# v3: Build multi-slot RAM cache [N, 6, 9, 224, 224] uint8
# ============================================================

dicom_root = Path(CFG['comp_input']) / CFG['dicom_subdir']
print(f'DICOM root: {dicom_root}')

# ---- Build slot mapping ----
# This calls the function defined in cell 04 (slot_matching.py)
slot_map, study_series_map = build_study_slot_map(series_meta, dicom_root)

# ---- Collect all studies needed for caching ----
# Training studies + validation studies
all_needed_uids = set(train_labels.index) | set(val_labels.index)
print(f'Studies to cache: {len(all_needed_uids)}')

# Build slot map for needed studies
# Slot map gives us which series to read for each slot
needed_slot_map = {}
for uid in all_needed_uids:
    if uid in slot_map:
        needed_slot_map[uid] = slot_map[uid]

# ---- Pre-allocate cache ----
n_cache_studies = len(needed_slot_map)
cache_shape = (n_cache_studies, N_SLOT, CFG['cache_slices'], CFG['image_size'], CFG['image_size'])
SLOT_CACHE = np.zeros(cache_shape, dtype=np.uint8)
SLOT_MASK = np.zeros((n_cache_studies, N_SLOT), dtype=np.float32)
study_index = {}  # {study_uid: row_index}

print(f'Cache: {cache_shape} = {SLOT_CACHE.nbytes / 1024**3:.2f} GB uint8')

# ---- Fill cache ----
t_cache = time.time()
completed = 0
failed = 0

for row_idx, study_uid in enumerate(sorted(needed_slot_map)):
    study_index[study_uid] = row_idx
    study_slots = needed_slot_map[study_uid]

    for slot_idx, (slot_name, plane, fluid, fatsat) in enumerate(SLOTS):
        slot_info = study_slots.get(slot_name)
        if slot_info is None:
            continue  # slot stays zero (mask already 0)

        series_dir = Path(slot_info['dir']) if 'dir' in slot_info else None
        if series_dir is None or not series_dir.exists():
            continue

        try:
            volume, px = read_series_volume(
                str(series_dir), plane=plane, image_size=CFG['image_size'])
            if volume is None or volume.shape[0] < 3:
                continue

            # Sample 9 slices from central 60%
            sampled = sample_cache_slices(
                volume, n_cache=CFG['cache_slices'], center_pct=CFG['center_pct'])

            # Laterality normalization
            laterality = None  # simplified: skip laterality for now
            if laterality and plane:
                for s in range(sampled.shape[0]):
                    sampled[s] = normalise_laterality(sampled[s], plane, laterality)

            # Convert to uint8 [0, 255]
            sampled_uint8 = (sampled * 255).clip(0, 255).round().astype(np.uint8)
            SLOT_CACHE[row_idx, slot_idx] = sampled_uint8
            SLOT_MASK[row_idx, slot_idx] = 1.0

        except Exception:
            failed += 1
            continue

    completed += 1
    if completed % 200 == 0:
        elapsed = time.time() - t_cache
        eta = (elapsed / completed) * (n_cache_studies - completed) / 60
        print(f'  [{completed:4d}/{n_cache_studies}] {elapsed:.0f}s | ~{eta:.0f}min remaining')

cache_time = time.time() - t_cache
total_series = int(SLOT_MASK.sum())
print(f'\nCache built: {n_cache_studies} studies, {total_series} series, '
      f'{SLOT_CACHE.nbytes / 1024**3:.1f} GB in {cache_time:.0f}s')
print(f'  Avg slots/study: {total_series/n_cache_studies:.1f}')
print(f'  Failed reads: {failed}')
gc.collect()



## 11. DataLoader


In [ ]:
# ============================================================
# v3: DataLoaders — train + val
# ============================================================

# Training: all studies return soft-label format (gold → prob=hard, weight=1, mask=1)
train_ds = MultiViewDataset(
    study_uids=train_labels.index.values,
    slot_map=slot_map,
    cache=SLOT_CACHE,
    mask_array=SLOT_MASK,
    labels_df=train_labels,
    study_index=study_index,
    is_train=True,
)

# Validation: gold studies return hard labels
val_ds = MultiViewDataset(
    study_uids=val_labels.index.values,
    slot_map=slot_map,
    cache=SLOT_CACHE,
    mask_array=SLOT_MASK,
    labels_df=val_labels,
    study_index=study_index,
    is_train=False,
)

# DataLoader config
loader_kw = dict(
    num_workers=CFG['num_workers'], pin_memory=True,
    prefetch_factor=2,
    persistent_workers=True if CFG['num_workers'] > 0 else False,
)
train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True, **loader_kw)
val_loader = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False, **loader_kw)

if IS_MAIN:
    print(f'Train batches: {len(train_loader):,}  (batch={CFG["batch_size"]}, '
          f'grad_accum={CFG["grad_accum_steps"]})')
    print(f'Val batches:   {len(val_loader):,}')
    eff_batch = CFG['batch_size'] * max(N_GPUS, 1) * CFG['grad_accum_steps']
    print(f'Effective batch: {eff_batch}')



## 12. 训练与验证函数


In [ ]:
# ============================================================
# v3: Training & Validation Functions
# ============================================================

def train_epoch(model, loader, optimizer, criterion, scaler, epoch):
    """v3: Unified training — all samples use soft-label format.

    Gold studies: prob = hard label (0/1), weight = 1.0, mask = 1.0
    Pseudo studies: prob = calibrated, weight = reliability, mask = thresholded
    Both use the same WeightedSoftBCELoss.
    """
    model.train()
    total_loss = 0.0
    n_batches = 0
    optimizer.zero_grad()
    use_amp = scaler is not None
    grad_accum = CFG.get('grad_accum_steps', 1)

    for bi, batch in enumerate(loader):
        slots = batch['slots'].to(DEVICE, non_blocking=True)
        mask = batch['mask'].to(DEVICE, non_blocking=True)
        prob_targets = batch['prob_targets'].to(DEVICE, non_blocking=True)
        weights = batch['weights'].to(DEVICE, non_blocking=True)
        soft_masks = batch['soft_masks'].to(DEVICE, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=use_amp):
            logits = model(slots, mask)
            loss = criterion(logits, prob_targets, weights, soft_masks)
            loss = loss / grad_accum

        if use_amp:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if (bi + 1) % grad_accum == 0:
            if use_amp:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
                optimizer.step()
            optimizer.zero_grad()

        total_loss += loss.item() * grad_accum
        n_batches += 1

        if IS_MAIN and bi % 20 == 0:
            slots_present = mask.sum(dim=1).mean().item()
            print(f'  Epoch {epoch:3d} [{bi:4d}/{len(loader):4d}] '
                  f'loss={loss.item()*grad_accum:.4f} | slots={slots_present:.1f}/6',
                  flush=True)

    return total_loss / max(n_batches, 1)


@torch.no_grad()
def validate_epoch(model, loader, criterion_hard):
    """Study-level validation on gold labels with partial-label support.

    Each gold study may have only a subset of the 12 targets labeled.
    Loss and per-class AUC are computed only on labeled targets.
    """
    model.eval()

    all_logits = []
    all_labels = []
    all_masks = []
    all_uids = []
    total_loss = 0.0
    n_batches = 0

    for batch in loader:
        slots = batch['slots'].to(DEVICE, non_blocking=True)
        mask = batch['mask'].to(DEVICE, non_blocking=True)
        labels = batch['labels'].to(DEVICE, non_blocking=True)
        val_masks = batch['val_masks'].to(DEVICE, non_blocking=True)
        uids = batch['study_uid']

        logits = model(slots, mask)

        # Masked BCE: only compute loss on labeled targets
        active = val_masks > 0.5
        if active.any():
            loss_val = F.binary_cross_entropy_with_logits(
                logits[active], labels[active], reduction='mean')
            total_loss += loss_val.item()
        n_batches += 1

        all_logits.append(logits.cpu())
        all_labels.append(labels.cpu())
        all_masks.append(val_masks.cpu())
        all_uids.extend(uids)

    # Concatenate
    logits_all = torch.cat(all_logits, dim=0).numpy()  # [N_val, 12]
    labels_all = torch.cat(all_labels, dim=0).numpy()
    masks_all = torch.cat(all_masks, dim=0).numpy()
    probs_all = 1.0 / (1.0 + np.exp(-logits_all))  # sigmoid

    # Per-class metrics — computed on LABELED studies only per class
    per_class = {}
    aucs = []
    n_studies = len(all_uids)

    for i, c in enumerate(TARGET_COLUMNS):
        # Filter to studies where THIS target is labeled
        labeled_idx = masks_all[:, i] > 0.5
        n_labeled = int(labeled_idx.sum())

        metrics = {
            'auc': float('nan'), 'accuracy': float('nan'),
            'precision': float('nan'), 'recall': float('nan'),
            'f1': float('nan'), 'n_pos': 0, 'n_total': n_labeled,
        }

        # Skip classes with too few labeled studies for meaningful metrics
        if n_labeled <= 1:
            per_class[c] = metrics
            continue

        y_true = labels_all[:, i][labeled_idx]
        y_prob = probs_all[:, i][labeled_idx]
        n_pos = int(y_true.sum())
        n_neg = n_labeled - n_pos
        metrics['n_pos'] = n_pos

        if n_pos == 0 or n_neg == 0:
            # All same label → AUC undefined, but accuracy still meaningful
            y_pred_binary = (y_prob >= 0.5).astype(int)
            metrics['accuracy'] = float((y_true == y_pred_binary).mean())
            per_class[c] = metrics
            # Still track if we have enough for AUC
            if n_pos > 0 and n_neg > 0:
                try:
                    a = roc_auc_score(y_true, y_prob)
                    metrics['auc'] = float(a)
                except Exception:
                    pass
            continue

        try:
            a = roc_auc_score(y_true, y_prob)
            metrics['auc'] = float(a)
            aucs.append(a)
        except Exception:
            pass

        y_pred_binary = (y_prob >= 0.5).astype(int)
        tp = int(((y_pred_binary == 1) & (y_true == 1)).sum())
        fp = int(((y_pred_binary == 1) & (y_true == 0)).sum())
        fn = int(((y_pred_binary == 0) & (y_true == 1)).sum())
        tn = n_labeled - tp - fp - fn

        metrics['accuracy'] = float((tp + tn) / n_labeled)
        metrics['precision'] = float(tp / (tp + fp)) if (tp + fp) > 0 else 0.0
        metrics['recall'] = float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0
        metrics['f1'] = float(
            2 * metrics['precision'] * metrics['recall']
            / (metrics['precision'] + metrics['recall'])
        ) if (metrics['precision'] + metrics['recall']) > 0 else 0.0
        per_class[c] = metrics

    return {
        'loss': total_loss / max(n_batches, 1),
        'macro_auc': float(np.mean(aucs)) if aucs else 0.0,
        'per_class': per_class,
        'probs': probs_all,
        'labels': labels_all,
        'uids': all_uids,
    }


def print_validation_summary(val_metrics):
    print(f'\n  {"Class":<20s} {"AUC":>7s} {"Acc":>7s} {"Prec":>7s} {"Rec":>7s} {"F1":>7s} {"Pos":>5s}')
    print(f'  {"-"*20} {"-"*7} {"-"*7} {"-"*7} {"-"*7} {"-"*7} {"-"*5}')
    for c in TARGET_COLUMNS:
        m = val_metrics['per_class'][c]
        auc_str = f'{m["auc"]:.3f}' if not math.isnan(m['auc']) else '  N/A  '
        print(f'  {c:<20s} {auc_str:>7s} {m["accuracy"]:7.3f} {m["precision"]:7.3f} '
              f'{m["recall"]:7.3f} {m["f1"]:7.3f} {m["n_pos"]:5d}')
    print(f'  {"-"*20} {"-"*7} {"-"*7} {"-"*7} {"-"*7} {"-"*7} {"-"*5}')
    print(f'  {"Macro AUC":<20s} {val_metrics["macro_auc"]:7.3f}')
    print()


def save_validation_report(val_metrics, output_dir, epoch=None, is_best=False):
    out = Path(output_dir)
    rows = []
    for c in TARGET_COLUMNS:
        m = val_metrics['per_class'][c]
        rows.append({
            'class': c, 'auc': m['auc'], 'accuracy': m['accuracy'],
            'precision': m['precision'], 'recall': m['recall'],
            'f1': m['f1'], 'n_pos': m['n_pos'], 'n_total': m['n_total'],
        })

    report_df = pd.DataFrame(rows)
    report_df['macro_auc'] = val_metrics['macro_auc']
    report_df['val_loss'] = val_metrics['loss']

    tag = '_best' if is_best else f'_epoch{epoch}'
    report_path = out / f'validation_report{tag}.csv'
    report_df.to_csv(report_path, index=False)

    # Save per-study predictions
    study_rows = []
    for i, uid in enumerate(val_metrics['uids']):
        row = {'StudyInstanceUID': uid}
        for j, c in enumerate(TARGET_COLUMNS):
            row[f'true_{c}'] = int(val_metrics['labels'][i, j])
            row[f'pred_{c}'] = float(val_metrics['probs'][i, j])
        study_rows.append(row)
    preds_df = pd.DataFrame(study_rows)
    preds_path = out / f'validation_predictions{tag}.csv'
    preds_df.to_csv(preds_path, index=False)

    if IS_MAIN:
        print(f'  Report: {report_path}')
        print(f'  Predictions: {preds_path} ({len(study_rows)} studies)')

    return report_df



## 13. 构建模型与优化器


In [ ]:
# ============================================================
# v3: Build model, optimizer, scheduler
# ============================================================

if IS_MAIN: print('Loading DINOv2 backbone...')

dinov2_backbone = timm.create_model(
    CFG['dinov2_variant'], pretrained=True, num_classes=0,
    img_size=CFG['image_size'],
)

model = MultiViewModel(
    dinov2_model=dinov2_backbone,
    n_slots=N_SLOT,
    cls_dim=CFG['cls_dim'],
    n_classes=CFG['num_classes'],
    slot_hidden=CFG['slot_hidden'],
    dropout=CFG['dropout'],
    unfreeze_layers=CFG['unfreeze_layers'],
).to(DEVICE)

if N_GPUS > 1:
    model = nn.DataParallel(model)
    print(f'[Model] DataParallel across {N_GPUS} GPUs')

# Separate LR for backbone vs head
backbone_params = []
head_params = []
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if 'dinov2' in name:
        backbone_params.append(p)
    else:
        head_params.append(p)

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': CFG['backbone_lr']},
    {'params': head_params, 'lr': CFG['lr']},
], weight_decay=CFG['weight_decay'])

if IS_MAIN:
    n_backbone = sum(p.numel() for p in backbone_params)
    n_head = sum(p.numel() for p in head_params)
    print(f'Optimizer: backbone {n_backbone/1e6:.1f}M params @ lr={CFG["backbone_lr"]}')
    print(f'           head     {n_head/1e6:.1f}M params @ lr={CFG["lr"]}')

criterion = WeightedSoftBCELoss()
criterion_val = HardBCELoss()

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=CFG['lr_t0'], T_mult=CFG['lr_t_mult'], eta_min=CFG['lr_eta_min'])

scaler = torch.amp.GradScaler('cuda') if CFG['mixed_precision'] else None



## 14. 管线检查


In [ ]:
# ============================================================
# v3: Sanity check — verify multi-view pipeline
# ============================================================
if IS_MAIN:
    batch = next(iter(train_loader))

    print(f'Slots shape:      {batch["slots"].shape}')       # [B, 6, 3, 224, 224]
    print(f'Mask shape:       {batch["mask"].shape}')        # [B, 6]
    print(f'Prob targets:     {batch["prob_targets"].shape}') # [B, 12]
    print(f'Weights:          {batch["weights"].shape}')      # [B, 12]
    print(f'Soft masks:       {batch["soft_masks"].shape}')   # [B, 12]
    print(f'Study UIDs:       {batch["study_uid"][:3]}')

    m = batch['mask']
    slots_present = m.sum(dim=1)
    print(f'\nSlots present per study: min={slots_present.min().item():.0f} '
          f'max={slots_present.max().item():.0f} mean={slots_present.float().mean().item():.1f}')

    p = batch['prob_targets']
    w = batch['weights']
    sm = batch['soft_masks']
    print(f'Soft label stats:')
    print(f'  prob   in [{p.min():.3f}, {p.max():.3f}], mean={p.mean():.3f}')
    print(f'  weight in [{w.min():.3f}, {w.max():.3f}], mean={w.mean():.3f}')
    print(f'  mask   % active: {(sm == 1).float().mean()*100:.1f}%')

    # Forward pass
    with torch.no_grad():
        out = model(batch['slots'].to(DEVICE), batch['mask'].to(DEVICE))
    print(f'\nForward pass:')
    print(f'  Output shape: {out.shape}')  # [B, 12]
    print(f'  Output range: [{out.min().item():.3f}, {out.max().item():.3f}]')

    # Test loss
    loss = criterion(
        out,
        batch['prob_targets'].to(DEVICE),
        batch['weights'].to(DEVICE),
        batch['soft_masks'].to(DEVICE),
    )
    print(f'  Soft BCE loss: {loss.item():.4f}')

    # Verify model output is reasonable
    probs_test = torch.sigmoid(out).cpu().numpy()
    print(f'  Pred prob range: [{probs_test.min():.3f}, {probs_test.max():.3f}]')
    print(f'  Pred prob mean: {probs_test.mean():.3f}')
    print(f'\n  Pipeline OK — ready for training!')



## 15. 训练循环


In [ ]:
# ============================================================
# v3: Training Loop
# ============================================================
if IS_MAIN:
    steps_per_epoch = len(train_loader)
    eff_batch = CFG['batch_size'] * max(N_GPUS, 1) * CFG['grad_accum_steps']
    print(f'\n{"="*60}')
    print(f'v3 Training — Multi-View 6-Slot + SlotHead')
    print(f'Batch={CFG["batch_size"]} × {max(N_GPUS,1)} GPUs × {CFG["grad_accum_steps"]} accum = {eff_batch} eff')
    print(f'Image={CFG["image_size"]}² | Slots={N_SLOT} | Feature dim={CFG["feature_dim"]}')
    print(f'Train studies={len(train_ds):,} ({steps_per_epoch} steps) | Val studies={len(val_ds):,}')
    print(f'{"="*60}\n')

best_auc = 0.0
best_epoch = 0
patience = 0
ckpt_dir = Path(CFG['output_dir']) / 'checkpoints'
ckpt_dir.mkdir(parents=True, exist_ok=True)
t_start = time.time()

history = []

for epoch in range(1, CFG['epochs'] + 1):
    t0 = time.time()

    train_loss = train_epoch(model, train_loader, optimizer, criterion, scaler, epoch)

    # Free GPU memory before validation
    torch.cuda.empty_cache()
    gc.collect()

    val_metrics = validate_epoch(model, val_loader, criterion_val)

    torch.cuda.empty_cache()
    scheduler.step()

    if IS_MAIN:
        epoch_time = time.time() - t0
        elapsed = time.time() - t_start
        vram = torch.cuda.max_memory_allocated(DEVICE) / 1024**3
        torch.cuda.reset_peak_memory_stats(DEVICE)
        lr_now = optimizer.param_groups[0]['lr']

        print(f'\n-- Epoch {epoch:3d}/{CFG["epochs"]} --')
        print(f'  Train Loss: {train_loss:.4f}  |  Val Loss: {val_metrics["loss"]:.4f}')
        print(f'  Val Macro AUC: {val_metrics["macro_auc"]:.4f}  |  LR: {lr_now:.2e}')
        print(f'  Time: {epoch_time:.0f}s | {elapsed/60:.0f}min total | VRAM: {vram:.1f}GB')

        print_validation_summary(val_metrics)

        history.append({
            'epoch': epoch, 'train_loss': train_loss,
            'val_loss': val_metrics['loss'], 'macro_auc': val_metrics['macro_auc'],
        })

        # Checkpoint on improvement
        current_auc = val_metrics['macro_auc']

        if current_auc > best_auc + 0.0005:
            best_auc = current_auc
            best_epoch = epoch
            patience = 0
            state = model.module.state_dict() if N_GPUS > 1 else model.state_dict()
            ckpt_path = ckpt_dir / 'best_model.pt'
            torch.save(
                {'epoch': epoch, 'model': state, 'auc': best_auc,
                 'config': CFG, 'slots': SLOTS, 'targets': TARGET_COLUMNS},
                ckpt_path,
            )
            print(f'  >> Best model saved (AUC={best_auc:.4f})')
            save_validation_report(val_metrics, CFG['output_dir'], epoch=epoch, is_best=True)
        else:
            patience += 1
            if patience >= CFG['early_stop_patience']:
                print(f'\n  Early stopping triggered at epoch {epoch}')
                break

# Final Report
if IS_MAIN:
    total_time = time.time() - t_start
    print(f'\n{"="*60}')
    print(f'v3 Training Complete')
    print(f'  Multi-View 6-Slot + SlotHead')
    print(f'  Best Val Macro AUC: {best_auc:.4f} (epoch {best_epoch})')
    print(f'  Total Time: {total_time/3600:.1f} hours')
    print(f'{"="*60}')

    history_df = pd.DataFrame(history)
    history_df.to_csv(Path(CFG['output_dir']) / 'training_history.csv', index=False)
    print(f'\nOutput files in {CFG["output_dir"]}/:')
    print(f'  training_history.csv')
    print(f'  checkpoints/best_model.pt')
    print(f'  validation_report_best.csv')
    print(f'  validation_predictions_best.csv')
    print(f'{"="*60}')



## 16. 阈值决策


### 训练完成后，根据数据判断是否需要逐类阈值

下面这个 cell 会自动分析每个类别的最优决策阈值：

- **大部分类的最优阈值 ≈ 0.5**：说明多视角 + SlotHead 已经修复了模型的校准问题 → **不需要逐类阈值**
- **仍有很多类的最优阈值 < 0.35**：说明模型仍然偏保守 → **需要逐类阈值**

不用猜，让数据说话。



## 17. Gold 验证（全部 gold 研究 AUC）


In [ ]:
# ============================================================
# v3: Gold-study validation — 完全自包含，不依赖前面的 cell
# ============================================================
#
# 使用方式：在 Kaggle notebook 中直接运行本 cell 即可。
# 不需要先跑前面的 cell。本 cell 包含所有需要的定义。
#
# 前提条件：
#   1. Kaggle notebook 已添加 RSNA 竞赛数据 (train.csv, train_series.csv, DICOM)
#   2. Kaggle notebook 已添加 checkpoint dataset (easoncyy/rsna-knee-v3-checkpoint)
#      或者 checkpoint 在 /kaggle/working/checkpoints/best_model.pt（上次训练 session）
#
# 预期运行时间：~2-3 分钟
# ============================================================

# ---- 0. 环境准备 ----
!pip install -q timm pydicom opencv-python scikit-learn 2>/dev/null

import gc, math, os, sys, time
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
import pydicom
import cv2
from sklearn.metrics import roc_auc_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ---- 1. 配置 (mirrors cells_v3/03_config.py) ----
TARGET_COLUMNS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA',
    'Effusion', 'Synovitis', "Baker's",
    'Contusion', 'Fracture',
]

SLOTS = [
    ("SAG_FLUID_FS",   "Sagittal", True,  True),
    ("COR_FLUID_FS",   "Coronal",  True,  True),
    ("AX_FLUID_FS",    "Axial",    True,  True),
    ("SAG_FLUID_NOFS", "Sagittal", True,  False),
    ("COR_T1",         "Coronal",  False, False),
    ("SAG_T1",         "Sagittal", False, False),
]
N_SLOT = len(SLOTS)

SLOT_PRIORS = {
    "ACL": (0, 3, 5), "MCL": (1, 4),
    "Medial Meniscus": (0, 1, 3, 4), "Lateral Meniscus": (0, 1, 3, 4),
    "Medial OA": (1, 4, 5), "Lateral OA": (1, 4, 5),
    "PF OA": (0, 2, 5), "Effusion": (0, 2), "Synovitis": (0, 2),
    "Baker's": (0,), "Contusion": (0, 1, 2), "Fracture": (0, 1, 2, 4, 5),
}

IMAGE_SIZE   = 224
CACHE_SLICES = 9
GROUP_SIZE   = 3

# ---- 2. 模型定义 (mirrors cells_v3/06_model.py) ----

class SlotHead(nn.Module):
    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden
        prior = torch.zeros(n_out, n_slot)
        for target_name, slot_indices in SLOT_PRIORS.items():
            if target_name in TARGET_COLUMNS:
                prior[TARGET_COLUMNS.index(target_name), list(slot_indices)] = 0.55
        self.register_buffer("slot_prior", prior)

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb
        attention = (
            torch.einsum("bsh,oh->bos", h, self.query) / math.sqrt(self.hidden)
            + self.slot_prior.unsqueeze(0)
        )
        attention = attention.masked_fill(mask.unsqueeze(1) < 0.5, -1e4).softmax(-1)
        context = self.drop(torch.einsum("bos,bsh->boh", attention, h))
        return (context * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias


class MultiViewModel(nn.Module):
    def __init__(self, dinov2_model, n_slots=6, cls_dim=384, n_classes=12,
                 slot_hidden=256, dropout=0.2, unfreeze_layers=6):
        super().__init__()
        self.n_slots = n_slots
        self.cls_dim = cls_dim
        self.feature_dim = cls_dim * 3
        self.unfreeze_layers = unfreeze_layers
        self.dinov2 = dinov2_model
        n_blocks = len(self.dinov2.blocks)
        if unfreeze_layers > 0:
            for p in self.dinov2.parameters():
                p.requires_grad = False
            unfreeze_start = max(0, n_blocks - unfreeze_layers)
            for block in self.dinov2.blocks[unfreeze_start:]:
                for p in block.parameters():
                    p.requires_grad = True
            if hasattr(self.dinov2, 'norm'):
                for p in self.dinov2.norm.parameters():
                    p.requires_grad = True
        self.head = SlotHead(dim=self.feature_dim, n_slot=n_slots, n_out=n_classes,
                             hidden=slot_hidden, p=dropout)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def _extract_features(self, x_3ch):
        if self.unfreeze_layers > 0:
            features = self.dinov2.forward_features(x_3ch)
        else:
            with torch.no_grad():
                features = self.dinov2.forward_features(x_3ch)
        cls = features[:, 0, :]
        patches = features[:, 1:, :]
        mean_p = patches.mean(dim=1)
        k = max(1, patches.shape[1] // 8)
        focal = patches.topk(k, dim=1).values.mean(dim=1)
        return torch.cat([cls, mean_p, focal], dim=1)

    def forward(self, images, mask):
        B, S = images.shape[:2]
        x = images.reshape(B * S, 3, images.shape[-2], images.shape[-1])
        x = x.float().div_(255.0)
        x = (x - self.mean) / self.std
        features = self._extract_features(x)
        features = features.reshape(B, S, -1)
        return self.head(features, mask)


# ---- 3. DICOM 读取 (mirrors cells_v3/05_dicom_io.py) ----

PLANE_SORT_AXIS = {"Sagittal": 0, "Coronal": 1, "Axial": 2}
_DICOM_TAGS = [
    (0x0020, 0x0032), (0x0020, 0x0013), (0x0028, 0x0002), (0x0028, 0x0004),
    (0x0028, 0x0010), (0x0028, 0x0011), (0x0028, 0x0100), (0x0028, 0x0101),
    (0x0028, 0x0102), (0x0028, 0x0103), (0x0028, 0x0030), (0x0028, 0x1052),
    (0x0028, 0x1053), (0x7FE0, 0x0010),
]

def _get_slice_position(ds, plane=None):
    try:
        ipp = getattr(ds, "ImagePositionPatient", None)
        if ipp and len(ipp) >= 3:
            return float(ipp[PLANE_SORT_AXIS.get(plane, 2)])
    except Exception: pass
    try:
        sl = getattr(ds, "SliceLocation", None)
        if sl is not None: return float(sl)
    except Exception: pass
    try:
        return float(getattr(ds, "InstanceNumber", 0))
    except Exception: return 0.0

def read_series_volume(series_dir, plane=None, image_size=224):
    series_dir = Path(series_dir)
    dcm_paths = sorted(series_dir.glob("*.dcm"))
    if not dcm_paths: dcm_paths = sorted(series_dir.glob("*"))
    if not dcm_paths: return None, None
    slices_info = []
    for p in dcm_paths:
        try:
            ds = pydicom.dcmread(str(p), force=True, specific_tags=_DICOM_TAGS)
            pos = _get_slice_position(ds, plane)
            img = ds.pixel_array.astype(np.float32)
            slope = float(getattr(ds, "RescaleSlope", 1) or 1)
            intercept = float(getattr(ds, "RescaleIntercept", 0) or 0)
            img = img * slope + intercept
            slices_info.append((pos, img))
        except Exception: continue
    if not slices_info: return None, None
    slices_info.sort(key=lambda x: x[0])
    images = np.stack([img for _, img in slices_info], axis=0)
    v_low, v_high = np.percentile(images, 1.0), np.percentile(images, 99.0)
    images = np.clip(images, v_low, v_high)
    denom = max(v_high - v_low, 1e-6)
    images = (images - v_low) / denom
    resized = []
    for img in images:
        r = cv2.resize(img, (image_size, image_size), interpolation=cv2.INTER_LINEAR)
        resized.append(r)
    return np.stack(resized, axis=0).astype(np.float32), None

def sample_cache_slices(volume, n_cache=9, center_pct=(0.2, 0.8)):
    n_total = volume.shape[0]
    if n_total <= n_cache:
        indices = list(range(n_total))
        while len(indices) < n_cache: indices.append(indices[-1])
        return volume[np.array(indices)]
    low, high = int(center_pct[0] * (n_total - 1)), int(center_pct[1] * (n_total - 1))
    if high <= low: low, high = 0, n_total - 1
    indices = np.unique(np.linspace(low, high, n_cache).astype(int))
    while len(indices) < n_cache: indices = np.append(indices, indices[-1])
    return volume[indices[:n_cache]]


# ---- 4. Slot 匹配 (mirrors cells_v3/04_slot_matching.py) ----

def match_slots_for_study(study_series_df):
    slots_found = {}
    for slot_name, plane, fluid, fatsat in SLOTS:
        candidates = study_series_df[
            (study_series_df["Anatomical_Plane"] == plane)
            & (study_series_df["Fluid_Sensitive"] == (1 if fluid else 0))
            & (study_series_df["Fat_Suppression"] == (1 if fatsat else 0))
        ]
        if len(candidates) == 0 and not fluid:
            candidates = study_series_df[
                (study_series_df["Anatomical_Plane"] == plane)
                & (study_series_df["Fluid_Sensitive"] == 0)
            ]
        if len(candidates) > 0:
            best = candidates.sort_values("n_slices", ascending=False).iloc[0]
            slots_found[slot_name] = {
                "series_uid": best["SeriesInstanceUID"],
                "dir": best["dir"],
                "n_slices": int(best["n_slices"]),
                "plane": plane,
            }
        else:
            slots_found[slot_name] = None
    return slots_found

def build_study_slot_map(series_meta, dicom_root):
    df = series_meta.copy()
    df["StudyInstanceUID"] = df["StudyInstanceUID"].astype(str)
    df["SeriesInstanceUID"] = df["SeriesInstanceUID"].astype(str)
    dirs, n_slices_list = [], []
    for _, row in df.iterrows():
        d = str(dicom_root / row["StudyInstanceUID"] / row["SeriesInstanceUID"])
        dirs.append(d)
        if os.path.isdir(d):
            n_slices_list.append(len([f for f in os.listdir(d) if f.endswith(".dcm")]))
        else:
            n_slices_list.append(0)
    df["dir"] = dirs
    df["n_slices"] = n_slices_list
    for col in ["Fluid_Sensitive", "Fat_Suppression", "Anatomical_Plane"]:
        if col not in df.columns:
            raise KeyError(f"train_series.csv missing column: {col}")
    slot_map, study_series_map = {}, {}
    for study_uid, grp in df.groupby("StudyInstanceUID"):
        study_series_map[study_uid] = grp
        slot_map[study_uid] = match_slots_for_study(grp)
    return slot_map, study_series_map


# ============================================================
# 以下是实际验证逻辑
# ============================================================

print("=" * 60)
print("GOLD-STUDY VALIDATION")
print("=" * 60)

# ---- 5. 路径设置 ----
COMP_INPUT = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
DICOM_ROOT = COMP_INPUT / "train_series"

# ---- 6. 找所有 gold 研究 ----
train_meta = pd.read_csv(COMP_INPUT / "train.csv")
train_meta["StudyInstanceUID"] = train_meta["StudyInstanceUID"].astype(str)
label_cols = [c for c in TARGET_COLUMNS if c in train_meta.columns]
has_all_labels = train_meta[label_cols].notna().all(axis=1)
gold_df = train_meta[has_all_labels].copy()
gold_uids = sorted(gold_df["StudyInstanceUID"].unique())

gold_labels = gold_df[["StudyInstanceUID"] + label_cols].copy()
gold_labels = gold_labels.set_index("StudyInstanceUID")
for c in TARGET_COLUMNS:
    if c not in gold_labels.columns:
        gold_labels[c] = np.nan
gold_labels = gold_labels.apply(pd.to_numeric, errors="coerce")

print(f"Gold studies (all 12 labeled): {len(gold_uids)}")
n_pos_per_class = (gold_labels > 0).sum(axis=0)
print(f"Positives per class: min={int(n_pos_per_class.min())}, "
      f"max={int(n_pos_per_class.max())}, mean={n_pos_per_class.mean():.1f}")

# ---- 7. 构建 gold-only slot map ----
series_meta = pd.read_csv(COMP_INPUT / "train_series.csv")
series_meta["StudyInstanceUID"] = series_meta["StudyInstanceUID"].astype(str)
series_meta["SeriesInstanceUID"] = series_meta["SeriesInstanceUID"].astype(str)
gold_series = series_meta[series_meta["StudyInstanceUID"].isin(gold_uids)]
print(f"Series rows: {len(series_meta):,} -> {len(gold_series):,} (gold only)")

gold_slot_map, _ = build_study_slot_map(gold_series, DICOM_ROOT)
valid_gold_uids = sorted([u for u in gold_uids if u in gold_slot_map])
print(f"Gold studies with DICOM: {len(valid_gold_uids)}")

# ---- 8. 构建 gold-only 缓存 ----
n_gold = len(valid_gold_uids)
GOLD_CACHE = np.zeros(
    (n_gold, N_SLOT, CACHE_SLICES, IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8)
GOLD_MASK = np.zeros((n_gold, N_SLOT), dtype=np.float32)
gold_study_idx = {}

t0 = time.time()
completed, failed = 0, 0

for row_idx, study_uid in enumerate(valid_gold_uids):
    gold_study_idx[study_uid] = row_idx
    study_slots = gold_slot_map[study_uid]

    for slot_idx, (slot_name, plane, fluid, fatsat) in enumerate(SLOTS):
        slot_info = study_slots.get(slot_name)
        if slot_info is None:
            continue
        series_dir = Path(slot_info["dir"]) if "dir" in slot_info else None
        if series_dir is None or not series_dir.exists():
            continue
        try:
            volume, px = read_series_volume(str(series_dir), plane=plane, image_size=IMAGE_SIZE)
            if volume is None or volume.shape[0] < 3:
                continue
            sampled = sample_cache_slices(volume, n_cache=CACHE_SLICES)
            sampled_uint8 = (sampled * 255).clip(0, 255).round().astype(np.uint8)
            GOLD_CACHE[row_idx, slot_idx] = sampled_uint8
            GOLD_MASK[row_idx, slot_idx] = 1.0
        except Exception:
            failed += 1
            continue
    completed += 1
    if completed % 20 == 0:
        print(f"  [{completed:3d}/{n_gold}] {time.time()-t0:.0f}s", flush=True)

elapsed = time.time() - t0
n_series = int(GOLD_MASK.sum())
print(f"Gold cache: {n_gold} studies, {n_series} series, "
      f"{GOLD_CACHE.nbytes/1024**3:.2f} GB in {elapsed:.0f}s "
      f"(avg {n_series/max(n_gold,1):.1f} slots/study, {failed} failed)")
gc.collect()

# ---- 9. 加载 checkpoint ----
ckpt_candidates = [
    Path("/kaggle/working/checkpoints/best_model.pt"),             # 旧 session
    Path("/kaggle/input/rsna-knee-v3-checkpoint/best_model.pt"),   # 上传的 dataset
]
ckpt_path = None
for p in ckpt_candidates:
    if p.exists():
        ckpt_path = p
        break

if ckpt_path is None:
    raise FileNotFoundError(
        "Checkpoint not found!\n"
        f"  请确认以下之一存在:\n"
        f"  1. {ckpt_candidates[0]} (上次训练 session 还在)\n"
        f"  2. {ckpt_candidates[1]} (上传的 dataset)\n"
        "  如果是新 session，需在右侧 Add Input -> Dataset -> "
        "搜索 'easoncyy/rsna-knee-v3-checkpoint' 并添加。"
    )

print(f"\nCheckpoint: {ckpt_path}")
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
print(f"  epoch={ckpt.get('epoch')}  saved_val_auc={ckpt.get('auc', 0):.4f}")

# ---- 10. 构建模型 & 加载权重 ----
dinov2 = timm.create_model(
    "vit_small_patch14_dinov2.lvd142m", pretrained=True,
    num_classes=0, img_size=IMAGE_SIZE)

model = MultiViewModel(
    dinov2_model=dinov2, n_slots=N_SLOT, cls_dim=384, n_classes=12,
    slot_hidden=256, dropout=0.2, unfreeze_layers=6,
).to(DEVICE)

state_dict = ckpt["model"]
first_key = next(iter(state_dict))
if first_key.startswith("module."):
    state_dict = {k.replace("module.", "", 1): v for k, v in state_dict.items()}
    print("  Stripped DataParallel prefix")

missing, unexpected = model.load_state_dict(state_dict, strict=False)
if missing:
    print(f"  WARNING: {len(missing)} missing keys")
if unexpected:
    print(f"  WARNING: {len(unexpected)} unexpected keys")

model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f"  Model loaded: {n_params/1e6:.1f}M params")

# ---- 11. TTA 推理 ----

class GoldDataset(Dataset):
    def __init__(self, uids, cache_arr, mask_arr, labels_df, study_idx):
        self.uids = uids; self.cache = cache_arr; self.mask = mask_arr
        self.labels = labels_df; self.study_idx = study_idx
    def __len__(self): return len(self.uids)
    def __getitem__(self, idx):
        uid = self.uids[idx]; ri = self.study_idx[uid]
        slots = torch.from_numpy(self.cache[ri].copy())
        m = torch.from_numpy(self.mask[ri].copy())
        # 7 个 3-slice 窗口
        windows = torch.stack([slots[:, w:w+3] for w in range(CACHE_SLICES - 3 + 1)], dim=0)
        labs = torch.tensor(
            [float(self.labels.loc[uid, c]) if not pd.isna(self.labels.loc[uid, c]) else 0.0
             for c in TARGET_COLUMNS], dtype=torch.float32)
        return windows, m, labs, uid


ds = GoldDataset(valid_gold_uids, GOLD_CACHE, GOLD_MASK, gold_labels, gold_study_idx)
loader = DataLoader(ds, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)


@torch.no_grad()
def tta_evaluate(model, loader):
    model.eval()
    logits_list, labels_list, uids_list = [], [], []
    for windows, mask, labels, uids in loader:
        windows = windows.to(DEVICE, non_blocking=True)
        mask = mask.to(DEVICE, non_blocking=True)
        B, W = windows.shape[0], windows.shape[1]
        flat = windows.reshape(B * W, *windows.shape[2:])
        flat_mask = mask.unsqueeze(1).expand(B, W, -1).reshape(B * W, -1)
        logits = model(flat, flat_mask).reshape(B, W, -1).mean(dim=1)
        logits_list.append(logits.cpu()); labels_list.append(labels); uids_list.extend(uids)
    logits_arr = torch.cat(logits_list).numpy()
    labels_arr = torch.cat(labels_list).numpy()
    probs_arr = 1.0 / (1.0 + np.exp(-logits_arr))
    return logits_arr, labels_arr, probs_arr, uids_list


print("Running TTA inference (7 windows x {} studies)...".format(len(valid_gold_uids)))
t1 = time.time()
logits_arr, labels_arr, probs_arr, all_uids = tta_evaluate(model, loader)
print(f"  Done in {time.time()-t1:.1f}s")

# ---- 12. Per-class AUC ----
aucs = {}
for i, c in enumerate(TARGET_COLUMNS):
    yt, yp = labels_arr[:, i], probs_arr[:, i]
    n_pos = int(yt.sum()); n_neg = len(yt) - n_pos
    if n_pos == 0 or n_neg == 0:
        aucs[c] = float("nan")
    else:
        try: aucs[c] = float(roc_auc_score(yt, yp))
        except Exception: aucs[c] = float("nan")

valid_aucs = [v for v in aucs.values() if not math.isnan(v)]
macro = float(np.mean(valid_aucs)) if valid_aucs else float("nan")

print(f"\n{'='*65}")
print(f"  GOLD VALIDATION -- {len(valid_gold_uids)} studies, 7-window TTA")
print(f"{'='*65}")
print(f"  {'Class':<20s} {'AUC':>7s} {'Pos':>5s} {'Neg':>5s}")
print(f"  {'-'*20} {'-'*7} {'-'*5} {'-'*5}")
for i, c in enumerate(TARGET_COLUMNS):
    a = aucs[c]
    n_pos_ = int(labels_arr[:, i].sum())
    n_neg_ = len(labels_arr) - n_pos_
    auc_str = f"{a:.4f}" if not math.isnan(a) else "  N/A  "
    print(f"  {c:<20s} {auc_str:>7s} {n_pos_:5d} {n_neg_:5d}")
print(f"  {'-'*20} {'-'*7} {'-'*5} {'-'*5}")
print(f"  {'Macro AUC':<20s} {macro:7.4f}")

# ---- 13. 保存结果 ----
out = Path("/kaggle/working")

rows = []
for i, uid in enumerate(all_uids):
    row = {"StudyInstanceUID": uid}
    for j, c in enumerate(TARGET_COLUMNS):
        row[f"true_{c}"] = int(labels_arr[i, j])
        row[f"prob_{c}"] = float(probs_arr[i, j])
    rows.append(row)
pd.DataFrame(rows).to_csv(out / "gold_validation_predictions.csv", index=False)

auc_rows = [{"class": c, "auc": aucs[c], "n_pos": int(labels_arr[:, i].sum())}
            for i, c in enumerate(TARGET_COLUMNS)]
auc_df = pd.DataFrame(auc_rows)
auc_df.loc["macro_avg"] = ["macro_avg", macro, ""]
auc_df.to_csv(out / "gold_validation_auc.csv", index=False)

print(f"\nSaved:")
print(f"  {out / 'gold_validation_predictions.csv'}")
print(f"  {out / 'gold_validation_auc.csv'}")
print(f"\nDone. Macro AUC = {macro:.4f} on {len(valid_gold_uids)} gold studies.")



In [ ]:
# ============================================================
# Per-Class Threshold Analysis
# ============================================================
if IS_MAIN:
    preds_path = Path(CFG['output_dir']) / 'validation_predictions_best.csv'
    if preds_path.exists():
        preds = pd.read_csv(preds_path)

        print('=' * 70)
        print('FINAL THRESHOLD ANALYSIS (Best Epoch)')
        print('=' * 70)

        summary = []
        for c in TARGET_COLUMNS:
            y_true = preds[f'true_{c}'].values
            y_prob = preds[f'pred_{c}'].values
            n_pos = int(y_true.sum())
            if n_pos == 0: continue

            best_thr, best_f1 = 0.5, 0.0
            for thr in np.arange(0.01, 0.99, 0.01):
                y_pred = (y_prob >= thr).astype(int)
                f1 = f1_score(y_true, y_pred, zero_division=0)
                if f1 > best_f1:
                    best_f1, best_thr = f1, thr

            f1_default = f1_score(y_true, (y_prob >= 0.5).astype(int), zero_division=0)
            pred_mean = y_prob.mean()
            pct_05 = (y_prob > 0.5).mean() * 100

            need = 'YES' if best_thr < 0.35 else ('maybe' if best_thr < 0.45 else 'no')
            summary.append({
                'class': c, 'best_threshold': best_thr,
                'f1_at_0.5': f1_default, 'f1_at_best': best_f1,
                'pred_mean': pred_mean, 'pct_above_0.5': pct_05,
                'need_tuning': need,
            })

        summary_df = pd.DataFrame(summary)
        print(summary_df.to_string(index=False))

        n_need = (summary_df['need_tuning'] == 'YES').sum()
        n_maybe = (summary_df['need_tuning'] == 'maybe').sum()

        print(f'\n{"="*70}')
        print(f'VERDICT')
        print(f'{"="*70}')

        if n_need >= 6:
            print(f'{n_need}/12 classes NEED threshold tuning, {n_maybe} maybe.')
            print(f'RECOMMENDATION: Apply per-class optimal thresholds.')
        elif n_need >= 2 or n_maybe >= 3:
            print(f'{n_need} need tuning, {n_maybe} maybe.')
            print(f'RECOMMENDATION: Optional per-class thresholds for lagging classes.')
        else:
            print(f'Only {n_need} class(es) need tuning.')
            print(f'RECOMMENDATION: Multi-view fixed calibration. Use default threshold=0.5.')

        summary_df.to_csv(Path(CFG['output_dir']) / 'threshold_analysis.csv', index=False)
        print(f'\nAnalysis saved: threshold_analysis.csv')
    else:
        print('No validation predictions found. Run training first.')

